In [1]:

import sys

import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)


3.12.7 2.3.4 2.3.3 1.7.2 3.10.7


In [2]:

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 3039

data = pd.read_csv("data/spambase.csv")

feature_names = [c for c in data.columns if c != "spam"]
X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)


(4204, 57) 39.9 (3153, 57) (1051, 57)


In [3]:

from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)

bag = BaggingClassifier(
    DecisionTreeClassifier(),
    n_estimators=100,
    oob_score=True,
    random_state=SEED
)

rf = RandomForestClassifier(
    n_estimators=300,
    oob_score=True,
    random_state=SEED,
    n_jobs=-1
)

lr = make_pipeline(
    FunctionTransformer(np.log1p),
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)

knn = make_pipeline(
    FunctionTransformer(np.log1p),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=15)
)

ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)

members = [("lr", lr), ("knn", knn), ("tree", tree)]


SEED: 3039

# Part 1 — Bagging and Out-of-Bag Estimates
## 1.1 OOB Mask

In [4]:

def oob_mask(sample_indices, n_samples):
    """Return a boolean mask for rows not included in the bootstrap sample."""
    mask = np.ones(n_samples, dtype=bool)
    mask[np.asarray(sample_indices, dtype=int)] = False
    return mask


In [5]:

print(oob_mask([0, 2, 2], 5))
print(oob_mask([], 3))
print(oob_mask(np.array([3, 0]), 4))


[False  True False  True  True]
[ True  True  True]
[False  True  True False]


## 1.2 Out-of-Bag Share

In [6]:

bag.fit(X_train, y_train)

n = len(X_train)

oob_shares = []

for sample_indices in bag.estimators_samples_:
    mask = oob_mask(sample_indices, n)
    oob_shares.append(mask.mean())

mean_oob = np.mean(oob_shares)
mean_inbag = 1 - mean_oob

theoretical_oob = (1 - 1 / n) ** n
theoretical_inbag = 1 - theoretical_oob

print(f"Mean OOB share: {mean_oob:.4f}")
print(f"Mean in-bag share: {mean_inbag:.4f}")
print(f"Theoretical OOB share: {theoretical_oob:.4f}")
print(f"Theoretical in-bag share: {theoretical_inbag:.4f}")


Mean OOB share: 0.3680
Mean in-bag share: 0.6320
Theoretical OOB share: 0.3678
Theoretical in-bag share: 0.6322


Conclusion: The mean OOB share is 0.3680, which is very close to the theoretical value of 0.3678. The mean in-bag share is 0.6320, compared with the theoretical value of 0.6322. The results show that approximately 36.88% of training samples are not selected for each bootstrap sample. This agrees with the theoretical expectation. 

## 1.3 OOB Accuracy by Hand

In [7]:

votes = np.zeros((n, 2), dtype=int)

for tree_model, sample_indices in zip(
    bag.estimators_, bag.estimators_samples_
):
    mask = oob_mask(sample_indices, n)
    oob_indices = np.where(mask)[0]

    predictions = tree_model.predict(X_train[oob_indices])

    for row_idx, prediction in zip(oob_indices, predictions):
        votes[row_idx, int(prediction)] += 1

valid = votes.sum(axis=1) > 0

oob_predictions = np.argmax(votes[valid], axis=1)
manual_oob_accuracy = np.mean(oob_predictions == y_train[valid])

print(f"Manual OOB accuracy: {manual_oob_accuracy:.6f}")
print(f"Sklearn OOB accuracy: {bag.oob_score_:.6f}")
print(f"Rows with OOB votes: {valid.sum()} / {n}")


Manual OOB accuracy: 0.935617
Sklearn OOB accuracy: 0.935617
Rows with OOB votes: 3153 / 3153


Conclusion: The manually calculated OOB accuracy is 0.935617, which exactly matches th scikit-learn OOB score of 0.935617. All 3153 training samples received OOB votes. This confirms that our manual OOB voting procedure works correctly. 

## 1.4 Five Models on the Same Folds

In [8]:

from sklearn.model_selection import cross_validate

models = {
    "tree": tree,
    "bag": bag,
    "rf": rf,
    "lr": lr,
    "bag_lr": bag_lr
}

cv_scores = {}

for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train,
        cv=cv, scoring="accuracy"
    )["test_score"]

    cv_scores[name] = scores

results = pd.DataFrame(
    {name: scores for name, scores in cv_scores.items()},
    index=[f"Fold {i}" for i in range(1, 6)]
).T

results["Mean"] = results.iloc[:, :5].mean(axis=1)
results["Std"] = results.iloc[:, :5].std(axis=1, ddof=0)

display(results.round(4))


,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean,Std
tree,0.8938,0.9113,0.8970,0.8937,0.8889,0.8969,0.0076
bag,0.9303,0.9493,0.9350,0.9365,0.9238,0.9350,0.0084
rf,0.9398,0.9699,0.9477,0.9381,0.9349,0.9461,0.0126
lr,0.9176,0.9525,0.9414,0.9508,0.9286,0.9382,0.0133
bag_lr,0.9223,0.9540,0.9445,0.9508,0.9302,0.9404,0.0122


## 1.5 Paired Fold Differences

In [9]:

pairs = [
    ("bag - tree", "bag", "tree"),
    ("rf - tree", "rf", "tree"),
    ("bag_lr - lr", "bag_lr", "lr")
]

differences = {}

for label, model_a, model_b in pairs:
    differences[label] = cv_scores[model_a] - cv_scores[model_b]

diff_table = pd.DataFrame(
    differences,
    index=[f"Fold {i}" for i in range(1, 6)]
).T

diff_table["Mean"] = diff_table.iloc[:, :5].mean(axis=1)
diff_table["Std"] = diff_table.iloc[:, :5].std(axis=1, ddof=0)

display(diff_table.round(4))


,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean,Std
bag - tree,0.0365,0.0380,0.0380,0.0429,0.0349,0.0381,0.0027
rf - tree,0.0460,0.0586,0.0507,0.0444,0.0460,0.0492,0.0052
bag_lr - lr,0.0048,0.0016,0.0032,0.0000,0.0016,0.0022,0.0016


Conclusion: Bagging improved the accuracy of the Decision Tree by 3.81 percentage points on average. Random Forest showed an even larger improvement of 4.92 percwntage points compared to the single tree. This happens because combining multiple trees reduces the effect of individual tree errors and makes predictions more stable. 

For Logistic Regression, bagging improved accuracy by only 0.22 percentage points. The improvement was small because Logistic Regression is generally a more stable model and benefits less from bagging. In all three comparisons, the mean paired difference was great than its standard deviation. 

## 1.6 One Forest, Three Estimates

In [10]:

rf.fit(X_train, y_train)

oob_accuracy = rf.oob_score_
cv_accuracy = cv_scores["rf"].mean()

test_accuracy = rf.score(X_test, y_test)
n_test = len(y_test)

standard_error = np.sqrt(
    test_accuracy * (1 - test_accuracy) / n_test
)

rf_cv_std = cv_scores["rf"].std(ddof=0)
oob_cv_difference = abs(oob_accuracy - cv_accuracy)

print(f"OOB accuracy: {oob_accuracy:.4f}")
print(f"CV accuracy: {cv_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test standard error: {standard_error:.4f}")
print(f"CV standard deviation: {rf_cv_std:.4f}")
print(f"OOB-CV difference: {oob_cv_difference:.4f}")
print(f"OOB and CV agree: {oob_cv_difference <= rf_cv_std}")


OOB accuracy: 0.9470
CV accuracy: 0.9461
Test accuracy: 0.9448
Test standard error: 0.0070
CV standard deviation: 0.0126
OOB-CV difference: 0.0010
OOB and CV agree: True


Сonclusion: Random Forest achieved an OOB accuracy of 0.9470, a cross-validation accuracy of 0.9461, and a test  accuray of 0.9448. The difference between OOB and Cv accuracy was approximately 0.0010, which is smaller than the CV standard deviation of 0.0126. Therefore, the two estimate agree according to the assignment criterion. 

The test accuracy was also close ton both estimates, with a standard error of 0.0070. These results suggest that OOB accuracy is a reasonable alternative to cross-validation for estimating Random Forest performance on this dataset. 